In [0]:
df_silver = spark.table(
    "workspace.hdb_pyspark.silver_hdb_resale"
)

In [0]:
from pyspark.sql.functions import avg,sum,round,count

df_gold_town_year = (
    df_silver
    .groupby(
        "transaction_year",
        "town"
    )
    .agg(
        round(avg("resale_price"),2).alias("avg_resale_price"),
        round(avg("price_per_sqm"), 2).alias("avg_price_per_sqm"),
        count("*").alias("transaction_count")
    )
    .orderBy(
        "transaction_year",
        "town"
    )
)

In [0]:
df_gold_town_year.show(20, truncate=False)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,desc

year_window = (
    Window
    .partitionBy("transaction_year")
    .orderBy(desc("avg_resale_price"))
)

df_gold_ranked = (
    df_gold_town_year
    .withColumn(
        "price_rank",
        rank().over(year_window)
    )
)

In [0]:
df_gold_ranked.filter(
    df_gold_ranked["price_rank"] <= 5
).show(30, truncate = False)

In [0]:
(
    df_gold_ranked.write
.format("delta")
.mode("overwrite")
.saveAsTable("workspace.hdb_pyspark.gold_town_year_metrics")
)

In [0]:
spark.table(
    "workspace.hdb_pyspark.gold_town_year_metrics"
).show(10, truncate= False)